In [1]:
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F

run_id, run_ts = str(uuid.uuid4()), datetime.now(timezone.utc)
results = []

def log(table, test, column, failed, total, severity="error", detail=""):
    status = "PASS" if failed == 0 else ("WARN" if severity == "warn" else "FAIL")
    results.append((run_id, run_ts, table, test, column, int(failed), int(total), status, detail))

def t_row_count(table, min_rows=1):
    n = spark.table(table).count()
    log(table, "row_count", "", 0 if n >= min_rows else 1, n, detail=f"min {min_rows}")

def t_not_null(table, cols):
    df = spark.table(table); total = df.count()
    for c in cols:
        log(table, "not_null", c, df.filter(F.col(c).isNull()).count(), total)

def t_unique(table, cols, severity="error"):
    df = spark.table(table); total = df.count()
    dups = df.groupBy(*cols).count().filter("count > 1").count()
    log(table, "unique", ",".join(cols), dups, total, severity)

def t_ref(fact, fk, dim, pk):
    f = spark.table(fact).filter(F.col(fk).isNotNull()); d = spark.table(dim)
    orphans = f.join(d, f[fk] == d[pk], "left_anti").count()
    log(fact, "referential_integrity", fk, orphans, f.count(), detail=f"-> {dim}.{pk}")

def t_rule(table, name, condition, severity="warn"):
    df = spark.table(table)
    log(table, name, "", df.filter(condition).count(), df.count(), severity, detail=condition)

StatementMeta(, a2a5e286-0ed5-4e63-810e-2d2d2525ddb0, 3, Finished, Available, Finished, False)

In [4]:
for t in ["dim_date", "dim_customer", "dim_product", "dim_region", "fact_sales", "fact_inventory"]:
    t_row_count(t)

# Uniqueness
t_unique("dim_date", ["date_key"]); t_unique("dim_customer", ["customer_key"])
t_unique("dim_product", ["product_key"]); t_unique("dim_region", ["region_key"])
t_unique("fact_sales", ["order_line_id"])
t_unique("fact_inventory", ["snapshot_date_key", "product_key", "region_key"])
t_unique("lh_bronze.dbo.file_products", ["product_id"], severity="warn")   # known source issue

# Nulls
t_not_null("fact_sales", ["order_date_key", "customer_key", "product_key", "region_key", "sales", "qty_ordered"])
t_not_null("fact_inventory", ["snapshot_date_key", "product_key", "region_key", "qty_on_hand"])

# Referential integrity
t_ref("fact_sales", "order_date_key", "dim_date", "date_key")
t_ref("fact_sales", "ship_date_key", "dim_date", "date_key")
t_ref("fact_sales", "customer_key", "dim_customer", "customer_key")
t_ref("fact_sales", "product_key", "dim_product", "product_key")
t_ref("fact_sales", "region_key", "dim_region", "region_key")
t_ref("fact_inventory", "snapshot_date_key", "dim_date", "date_key")
t_ref("fact_inventory", "product_key", "dim_product", "product_key")
t_ref("fact_inventory", "region_key", "dim_region", "region_key")

# Reconciliation: silver order lines vs gold fact
src, tgt = spark.table("lh_silver.dbo.order_lines").count(), spark.table("fact_sales").count()
log("fact_sales", "reconcile_row_count", "", abs(src - tgt), src, detail=f"silver {src} / gold {tgt}")

# Business rules (warnings)
t_rule("fact_sales", "ship_before_order", "days_to_ship < 0")
t_rule("fact_sales", "over_shipped", "qty_shipped > qty_ordered")
t_rule("fact_sales", "missing_ship_date", "ship_date_key IS NULL")

StatementMeta(, a2a5e286-0ed5-4e63-810e-2d2d2525ddb0, 6, Finished, Available, Finished, False)

In [5]:
schema = ("run_id string, run_ts timestamp, table_name string, test_name string, column_name string, "
          "failed_rows long, total_rows long, status string, detail string")
spark.createDataFrame(results, schema).write.mode("append").saveAsTable("dq_results")

fails = [r for r in results if r[7] == "FAIL"]
print(f"{len(results)} tests, {len(fails)} failed, run {run_id}")
if fails:
    raise Exception(f"{len(fails)} data quality tests failed - see dq_results for run {run_id}")

StatementMeta(, a2a5e286-0ed5-4e63-810e-2d2d2525ddb0, 7, Finished, Available, Finished, False)

78 tests, 2 failed, run cd3d8d24-a66a-48e2-8376-6b271a3a327e


Exception: 2 data quality tests failed - see dq_results for run cd3d8d24-a66a-48e2-8376-6b271a3a327e